---
---
## Task 1 - K-means Unsupervised Classification
---
---
本任務使用 Sentinel-2 震後影像進行 K-means 非監督分類，目的是在沒有訓練樣本的情況下，先探索研究區內不同地物的光譜群集。前處理後的有效像素特徵矩陣 `X_valid` 包含 6 個波段：B02、B03、B04、B08、B11、B12。

工作流程如下：

1. 讀取前處理後的 `img_sr`, `X_valid`, `valid_mask`
2. 使用 `KMeans(n_clusters=5)` 將有效像素分成 5 個光譜群集
3. 將一維 cluster labels 還原成 2D classification map
4. 繪製 K-means 分類圖
5. 計算每個 cluster 的平均光譜值
6. 根據平均光譜曲線與影像判讀，手動將 cluster 對應到 Water、Forest、Cropland、Bare/Landslide、Built-up 等地物類別
7. 輸出 K-means classification map、cluster mean spectrum table 與光譜判讀說明


### Step 1 - 影像搜尋
---
透過 Microsoft Planetary Computer STAC API 搜尋秀林/太魯閣研究區的 Sentinel-2 L2A 震後影像，並使用多階段條件逐步放寬日期與雲量限制。最後依雲量排序，選出雲量最低且可用的 post-earthquake scene 作為後續分類分析的主要影像。

In [ ]:
import pystac_client
import planetary_computer as pc
import stackstac

catalog = pystac_client.Client.open(
    'https://planetarycomputer.microsoft.com/api/stac/v1',
    modifier=pc.sign_inplace,
)

# Xiulin / Taroko study area BBOX
# West: Central Mountain Range foothills; East: Pacific Ocean (ensures water samples)
TAROKO_BBOX = [121.40, 24.10, 121.80, 24.25]

# Post-earthquake imagery — progressive search (strict → relaxed)
# Hualien April-May = plum rain season → high cloud cover, need flexible strategy
search_configs = [
    ('2024-04-15/2024-05-31', 20, 'Phase 1: 2 months post-quake, CC < 20%'),
    ('2024-04-03/2024-08-31', 30, 'Phase 2: 5 months post-quake, CC < 30%'),
    ('2024-04-03/2024-12-31', 50, 'Phase 3: full year post-quake, CC < 50%'),
]

for dt_range, max_cc, desc in search_configs:
    print(f"Trying {desc}...")
    search = catalog.search(
        collections=['sentinel-2-l2a'],
        bbox=TAROKO_BBOX,
        datetime=dt_range,
        query={'eo:cloud_cover': {'lt': max_cc}},
    )
    items = list(search.items())
    print(f"  → Found {len(items)} scenes")
    if len(items) > 0:
        break

# Sort by cloud cover, pick the best scene
items_sorted = sorted(items, key=lambda x: x.properties['eo:cloud_cover'])
best_item = items_sorted[0]
print(f"Selected: {best_item.id}, cloud cover: {best_item.properties['eo:cloud_cover']:.1f}%")

將選定的 Sentinel-2 影像讀取為 20 m 解析度的多波段資料立方體，並取出 B02、B03、B04、B08、B11、B12 與 SCL 波段。接著將原始數值轉換為 surface reflectance，使用 SCL 保留清晰地表像素，建立 `X_valid` 特徵矩陣，並將前處理結果存成 `.npy` 檔供後續 K-means 與 Random Forest 分類使用。

In [ ]:
# Preprocess first, then save all outputs
import os
import json
import numpy as np
import stackstac

output_dir = r"c:\Users\Ben\Documents\NTU\RSGI\homework11\data"
os.makedirs(output_dir, exist_ok=True)

BANDS = ["B02", "B03", "B04", "B08", "B11", "B12"]
BANDS_ALL = BANDS + ["SCL"]

# 1) Build cube from best_item
cube = stackstac.stack(
    [best_item],
    assets=BANDS_ALL,
    bounds_latlon=TAROKO_BBOX,
    resolution=20,
    epsg=32651,
).isel(time=0).compute()

arr = cube.values  # shape: (7, h, w)

# 2) Preprocess
img_raw = arr[:6].astype("float32")      # B02..B12
scl = arr[6].astype("int16")             # SCL
img_sr = img_raw / 10000.0               # surface reflectance

# Keep clear pixels only: vegetation / non-vegetated / water
valid_scl = np.isin(scl, [4, 5, 6])

n_bands, h, w = img_sr.shape
X = img_sr.reshape(n_bands, -1).T        # (h*w, 6)
valid = (~np.isnan(X).any(axis=1)) & valid_scl.reshape(-1)
X_valid = X[valid]

print("img_sr:", img_sr.shape)
print("X:", X.shape)
print("X_valid:", X_valid.shape)
print("valid_ratio:", valid.mean())

# 3) Save AFTER preprocessing
with open(os.path.join(output_dir, "best_item.json"), "w", encoding="utf-8") as f:
    json.dump(best_item.to_dict(), f, ensure_ascii=False, indent=2)

np.save(os.path.join(output_dir, "img_sr.npy"), img_sr)
np.save(os.path.join(output_dir, "scl.npy"), scl)
np.save(os.path.join(output_dir, "X.npy"), X)
np.save(os.path.join(output_dir, "valid_mask.npy"), valid)
np.save(os.path.join(output_dir, "X_valid.npy"), X_valid)

with open(os.path.join(output_dir, "preprocess_meta.txt"), "w", encoding="utf-8") as f:
    f.write(f"img_sr_shape={img_sr.shape}\n")
    f.write(f"scl_shape={scl.shape}\n")
    f.write(f"X_shape={X.shape}\n")
    f.write(f"X_valid_shape={X_valid.shape}\n")
    f.write(f"valid_ratio={valid.mean():.4f}\n")
    f.write("kept_scl_classes=4,5,6\n")
    f.write("bands=B02,B03,B04,B08,B11,B12\n")
    f.write(f"item_id={best_item.id}\n")
    f.write(f"cloud_cover={best_item.properties.get('eo:cloud_cover')}\n")

print("Saved: best_item.json, img_sr.npy, scl.npy, X.npy, valid_mask.npy, X_valid.npy, preprocess_meta.txt")

### Step 2 - K-means (K=5)
---
使用前處理後的有效像素矩陣 `X_valid` 執行 K-means 分類，將研究區依光譜特徵分成 5 個群集。接著將分類結果還原為 2D 地圖，並輸出 K-means classification map 供後續光譜判讀與地物標籤使用。

In [ ]:
# Task 1 - Step 2: K-means (K=5) + rebuild 2D map
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
input_dir = r"c:\Users\Ben\Documents\NTU\RSGI\homework11\data"
output_dir = r"c:\Users\Ben\Documents\NTU\RSGI\homework11\outputs\Task1"
os.makedirs(output_dir, exist_ok=True)

# Load saved preprocessing outputs
X_valid = np.load(os.path.join(input_dir, "X_valid.npy"))
valid = np.load(os.path.join(input_dir, "valid_mask.npy"))
img_sr = np.load(os.path.join(input_dir, "img_sr.npy"))

_, h, w = img_sr.shape

# KMeans
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
labels = kmeans.fit_predict(X_valid)  # shape: (n_valid_pixels,)

# Rebuild 2D label map
label_map = np.full(h * w, np.nan, dtype="float32")
label_map[valid] = labels
label_map = label_map.reshape(h, w)

# Plot
# label_map 裡有 NaN，把 NaN mask 起來
plot_map = np.ma.masked_invalid(label_map)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(plot_map, cmap="tab10")
ax.set_title("K-means Classification (K=5)")
ax.axis("off")
cbar = fig.colorbar(im, ax=ax, label="Cluster ID")

fig.savefig(os.path.join(output_dir, "kmeans_classification.png"), dpi=300, bbox_inches="tight")
plt.close(fig)

# Save outputs for next step/report
np.save(os.path.join(output_dir, "kmeans_labels.npy"), labels)
np.save(os.path.join(output_dir, "kmeans_label_map.npy"), label_map)

print("Saved: kmeans_labels.npy, kmeans_label_map.npy, kmeans_classification.png")

### Step 3 - K-means output
---
計算每個 K-means cluster 在六個 Sentinel-2 波段上的平均光譜，並根據光譜曲線與地物特徵手動對應為 Water、Forest、Cropland、Bare/Landslide 與 Built-up。最後輸出標籤化分類圖、cluster-to-class 對照表，以及平均光譜折線圖，用於解釋各群集的地物意義。

In [ ]:
# Task 1 - Step 3: mean spectrum + manual cluster labeling + labeled map
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
output_dir = r"c:\Users\Ben\Documents\NTU\RSGI\homework11\outputs\Task1"
n_clusters = 5

# 1) Compute mean spectrum per cluster
rows = []
for k in range(n_clusters):
    cluster_mean = X_valid[labels == k].mean(axis=0)
    rows.append([k, *cluster_mean])
    print(
        f"Cluster {k}: "
        f"B02={cluster_mean[0]:.3f}, B03={cluster_mean[1]:.3f}, B04={cluster_mean[2]:.3f}, "
        f"B08={cluster_mean[3]:.3f}, B11={cluster_mean[4]:.3f}, B12={cluster_mean[5]:.3f}"
    )

df_cluster = pd.DataFrame(rows, columns=["cluster"] + BANDS_ALL[:-1])
display(df_cluster.round(4))

# 2) Manual mapping (edit this after you inspect spectra + map)
# class_id: 1=Water, 2=Forest, 3=Cropland, 4=Bare/Landslide, 5=Built-up
cluster_to_class = {
    0: 2,  # example: Forest
    1: 4,  # example: Bare/Landslide
    2: 1,  # example: Water
    3: 5,  # example: Built-up
    4: 3,  # example: Cropland
}

class_names = {
    1: "Water",
    2: "Forest",
    3: "Cropland",
    4: "Bare/Landslide",
    5: "Built-up"
}

# 3) Convert cluster labels -> class labels (pixel list)
class_labels = np.array([cluster_to_class[int(k)] for k in labels], dtype=np.int16)

# 4) Rebuild 2D class map using valid mask
class_map = np.full(h * w, np.nan, dtype="float32")
class_map[valid] = class_labels
class_map = class_map.reshape(h, w)

# 5) Plot labeled map
colors = ['#1f78b4', '#33a02c', '#b2df8a', '#b15928', '#6a3d9a']  # 1..5
cmap = ListedColormap(colors)

plt.figure(figsize=(10, 8))
im = plt.imshow(class_map, cmap=cmap, vmin=1, vmax=5)
cbar = plt.colorbar(im, ticks=[1,2,3,4,5])
cbar.ax.set_yticklabels([class_names[i] for i in [1,2,3,4,5]])
plt.title("K-means Labeled Land Cover Map")
plt.axis("off")
plt.show()

# 6) Save outputs
df_cluster.to_csv(fr"{output_dir}\kmeans_cluster_mean_spectrum.csv", index=False)
pd.DataFrame({
    "cluster": list(cluster_to_class.keys()),
    "class_id": list(cluster_to_class.values()),
    "class_name": [class_names[v] for v in cluster_to_class.values()]
}).to_csv(fr"{output_dir}\kmeans_cluster_to_class.csv", index=False)

np.save(fr"{output_dir}\kmeans_class_labels.npy", class_labels)
np.save(fr"{output_dir}\kmeans_class_map.npy", class_map)

plt.imsave(fr"{output_dir}\kmeans_labeled_map.png", class_map, cmap=cmap, vmin=1, vmax=5)
print("Saved Task1 Step3 outputs.")

plt.figure(figsize=(9, 6))
for k in range(5):
    m = X_valid[labels == k].mean(axis=0)
    plt.plot(BANDS_ALL[:-1], m, marker="o", linewidth=2, label=f"Cluster {k}")

plt.title("Mean Spectral Signature per K-means Cluster")
plt.xlabel("Band")
plt.ylabel("Reflectance")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(fr"{output_dir}\kmeans_cluster_mean_spectrum.png", dpi=300)
plt.show()

### Brief discussion:
---
    最容易解讀的通常是 Water cluster。
    原因是水體在光譜上特徵很明顯：B08 (NIR) 和 B11/B12 (SWIR) 反射率都很低，通常會和其他地物清楚分開，所以在 K-means 裡最容易形成穩定、可辨識的一群。在本次實作中water cluster是cluster 2。

---
---
## Task 2: Random Forest Supervised Classification
---
---
本任務使用 Random Forest 監督式分類方法，根據 Google Earth 手動標記的訓練樣本建立土地覆蓋分類模型。訓練類別包含 Water、Forest、Cropland、Bare/Landslide 與 Built-up 五類，並使用 Sentinel-2 的六個反射率波段作為模型特徵。

工作流程如下：

1. 讀取 Google Earth 匯出的訓練樣本 KMZ，並將 polygon ROI 轉換為影像像素座標
2. 從 `img_sr` 中擷取訓練像素的六波段光譜值，建立 `X_train` 與 `y_train`
3. 將樣本切分為 80% 訓練資料與 20% 測試資料
4. 訓練 `RandomForestClassifier`，並計算 Training accuracy、Test accuracy 與 OOB accuracy
5. 使用訓練完成的模型分類整張有效影像，產生 Random Forest land cover map
6. 計算 feature importance，分析哪些 Sentinel-2 波段對分類最重要
7. 將 K-means 與 Random Forest 結果並排比較，檢視非監督與監督式分類差異

### Step 1 - Read KMZ polygons and convert to training_data row/col samples
---
讀取 Google Earth 匯出的訓練樣本 KMZ，解析各類別 polygon，並將其從經緯度座標轉換到 Sentinel-2 影像使用的 UTM 座標系。接著將 polygon rasterize 成影像格網，擷取每個類別內的有效像素 row/col，建立 Random Forest 訓練用的 `training_data`。


In [ ]:
# Task 2 - Step 1: Read KMZ polygons and convert to training_data row/col samples

import os
import zipfile
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd

from shapely.geometry import Polygon
from shapely.ops import transform as shapely_transform
from pyproj import Transformer
from rasterio.features import rasterize
from rasterio.transform import from_bounds

input_dir = r"c:\Users\Ben\Documents\NTU\RSGI\homework11\data"
output_dir = r"c:\Users\Ben\Documents\NTU\RSGI\homework11\outputs\Task2"
os.makedirs(output_dir, exist_ok=True)

kmz_path = os.path.join(input_dir, "trainning_data.kmz")  # your current filename
img_sr = np.load(os.path.join(input_dir, "img_sr.npy"))
scl = np.load(os.path.join(input_dir, "scl.npy"))

_, h, w = img_sr.shape
TAROKO_BBOX = [121.40, 24.10, 121.80, 24.25]

class_names = ["Water", "Forest", "Cropland", "Bare", "Built-up"]

def infer_class(name):
    name_low = name.lower()
    if "water" in name_low:
        return "Water"
    if "forest" in name_low:
        return "Forest"
    if "cropland" in name_low or "corpland" in name_low:
        return "Cropland"
    if "bare" in name_low:
        return "Bare"
    if "built" in name_low:
        return "Built-up"
    return None

# 1) Read doc.kml from KMZ
with zipfile.ZipFile(kmz_path, "r") as z:
    kml_name = [n for n in z.namelist() if n.endswith(".kml")][0]
    kml_text = z.read(kml_name).decode("utf-8")

root = ET.fromstring(kml_text)
ns = {"kml": "http://www.opengis.net/kml/2.2"}

polygons_by_class = {name: [] for name in class_names}

def direct_name(el):
    name_el = el.find("kml:name", ns)
    return name_el.text.strip() if name_el is not None and name_el.text else ""

def parse_folder(folder, current_class=None):
    folder_name = direct_name(folder)
    folder_class = infer_class(folder_name) or current_class

    for pm in folder.findall("kml:Placemark", ns):
        coords_el = pm.find(".//kml:coordinates", ns)
        if coords_el is None or folder_class is None:
            continue

        coords = []
        for pt in coords_el.text.strip().split():
            lon, lat, *_ = pt.split(",")
            coords.append((float(lon), float(lat)))

        if len(coords) >= 3:
            polygons_by_class[folder_class].append(Polygon(coords))

    for subfolder in folder.findall("kml:Folder", ns):
        parse_folder(subfolder, folder_class)

for folder in root.findall(".//kml:Folder", ns):
    parse_folder(folder)

print("Polygons found:")
for name in class_names:
    print(f"{name}: {len(polygons_by_class[name])}")

# 2) Build raster transform matching EPSG:32651 image grid
transformer = Transformer.from_crs("EPSG:4326", "EPSG:32651", always_xy=True)

xmin, ymin = transformer.transform(TAROKO_BBOX[0], TAROKO_BBOX[1])
xmax, ymax = transformer.transform(TAROKO_BBOX[2], TAROKO_BBOX[3])

# Snap bounds to 20 m grid, matching stackstac style
xmin = np.floor(xmin / 20) * 20
ymin = np.floor(ymin / 20) * 20
xmax = xmin + w * 20
ymax = ymin + h * 20

transform = from_bounds(xmin, ymin, xmax, ymax, w, h)

# 3) Rasterize each class polygon and extract row/col
training_data = {}

for class_name in class_names:
    geoms_utm = [
        shapely_transform(transformer.transform, geom)
        for geom in polygons_by_class[class_name]
    ]

    mask = rasterize(
        [(geom, 1) for geom in geoms_utm],
        out_shape=(h, w),
        transform=transform,
        fill=0,
        dtype="uint8",
    )

    rows, cols = np.where(mask == 1)

    # Keep only valid pixels: avoid cloud/shadow/NaN
    good = []
    for r, c in zip(rows, cols):
        pixel = img_sr[:, r, c]
        if not np.isnan(pixel).any() and scl[r, c] in [4, 5, 6]:
            good.append((r, c))

    rows_clean = [r for r, c in good]
    cols_clean = [c for r, c in good]

    training_data[class_name] = {
        "rows": rows_clean,
        "cols": cols_clean,
    }

# 4) Summary table
summary = []
for class_name in class_names:
    n = len(training_data[class_name]["rows"])
    summary.append({"class": class_name, "n_pixels": n})
    print(f"{class_name}: {n} training pixels")

df_summary = pd.DataFrame(summary)
display(df_summary)

df_summary.to_csv(os.path.join(output_dir, "training_sample_summary.csv"), index=False)
np.save(os.path.join(output_dir, "training_data.npy"), training_data, allow_pickle=True)

print("Saved: training_sample_summary.csv, training_data.npy")

### Step 2 - Train Random Forest
---
將 `training_data` 中的 row/col 樣本轉換為六波段光譜特徵與類別標籤，建立 Random Forest 的訓練資料集。接著將樣本切分為 80% 訓練資料與 20% 測試資料，訓練 `RandomForestClassifier`，並輸出 training accuracy、test accuracy 與 OOB accuracy 作為模型初步評估。

In [ ]:
# Task 2 - Step 2: Train Random Forest
import os
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

input_dir = r"c:\Users\Ben\Documents\NTU\RSGI\homework11\data"
task2_dir = r"c:\Users\Ben\Documents\NTU\RSGI\homework11\outputs\Task2"

img_sr = np.load(os.path.join(input_dir, "img_sr.npy"))

training_data = np.load(
    os.path.join(task2_dir, "training_data.npy"),
    allow_pickle=True
).item()

X_train_list, y_train_list = [], []
class_names = list(training_data.keys())

for class_id, class_name in enumerate(class_names):
    roi = training_data[class_name]
    for r, c in zip(roi["rows"], roi["cols"]):
        pixel = img_sr[:, r, c]  # img_sr is numpy array
        if not np.isnan(pixel).any():
            X_train_list.append(pixel)
            y_train_list.append(class_id)

X_train = np.array(X_train_list)
y_train = np.array(y_train_list)

X_tr, X_te, y_tr, y_te = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    oob_score=True
)

rf.fit(X_tr, y_tr)

print("class_names:", class_names)
print("X_train:", X_train.shape)
print(f"Training accuracy: {rf.score(X_tr, y_tr):.3f}")
print(f"Test accuracy: {rf.score(X_te, y_te):.3f}")
print(f"OOB accuracy: {rf.oob_score_:.3f}")

### Step 3 - Classify entire image with Random Forest
---
使用訓練完成的 Random Forest 模型對整張影像的有效像素 `X_valid` 進行預測，產生每個像素的土地覆蓋類別。接著將一維預測結果還原為 2D classification map，並輸出 Random Forest 土地覆蓋分類圖與分類結果陣列。

In [ ]:
# Task 2 - Step 3: Classify entire image with Random Forest

import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

input_dir = r"c:\Users\Ben\Documents\NTU\RSGI\homework11\data"
output_dir = r"c:\Users\Ben\Documents\NTU\RSGI\homework11\outputs\Task2"

# Load full valid feature matrix and valid mask
X_valid = np.load(os.path.join(input_dir, "X_valid.npy"))
valid = np.load(os.path.join(input_dir, "valid_mask.npy"))
img_sr = np.load(os.path.join(input_dir, "img_sr.npy"))

_, h, w = img_sr.shape

# Predict all valid pixels
y_pred_all = rf.predict(X_valid)

# Rebuild 2D classification map
class_map = np.full(h * w, np.nan, dtype="float32")
class_map[valid] = y_pred_all
class_map = class_map.reshape(h, w)

# Class color map
# 0=Water, 1=Forest, 2=Cropland, 3=Bare, 4=Built-up
colors = ["#0077BE", "#228B22", "#DAA520", "#CD853F", "#808080"]
cmap = ListedColormap(colors)

# Plot
fig, ax = plt.subplots(figsize=(12, 10))
plot_map = np.ma.masked_invalid(class_map)

im = ax.imshow(plot_map, cmap=cmap, vmin=0, vmax=4)
ax.set_title("Random Forest Land Cover Classification")
ax.axis("off")

cbar = fig.colorbar(im, ax=ax, ticks=range(5), label="Class")
cbar.ax.set_yticklabels(class_names)

plt.show()

# Save outputs
np.save(os.path.join(output_dir, "rf_class_map.npy"), class_map)
np.save(os.path.join(output_dir, "rf_y_pred_all.npy"), y_pred_all)

fig.savefig(
    os.path.join(output_dir, "rf_classification.png"),
    dpi=300,
    bbox_inches="tight"
)

print("Saved: rf_class_map.npy, rf_y_pred_all.npy, rf_classification.png")

### Step 4 - Feature importance
---
計算 Random Forest 模型中各 Sentinel-2 波段的 feature importance，評估哪些光譜波段對土地覆蓋分類貢獻最大。結果以長條圖與表格輸出，用於解釋模型判斷依據，例如 B11 對區分裸地、水體與植被具有較高重要性。

In [ ]:
# Task 2 - Step 4: Feature Importance
import os
import pandas as pd
import matplotlib.pyplot as plt

output_dir = r"c:\Users\Ben\Documents\NTU\RSGI\homework11\outputs\Task2"
os.makedirs(output_dir, exist_ok=True)

importance = rf.feature_importances_
band_names = ['B02', 'B03', 'B04', 'B08', 'B11', 'B12']

df_importance = pd.DataFrame({
    "band": band_names,
    "importance": importance
}).sort_values("importance", ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(df_importance["band"], df_importance["importance"])
ax.set_xlabel("Feature Importance")
ax.set_title("Random Forest - Band Importance")
plt.tight_layout()
plt.show()

fig.savefig(os.path.join(output_dir, "rf_feature_importance.png"), dpi=300, bbox_inches="tight")
df_importance.to_csv(os.path.join(output_dir, "rf_feature_importance.csv"), index=False)

display(df_importance.sort_values("importance", ascending=False))
print("Saved: rf_feature_importance.png, rf_feature_importance.csv")

### Step 5 - Compare classification maps
---
將 K-means 非監督分類結果與 Random Forest 監督式分類結果並排呈現，方便比較兩種方法在空間分布與類別判讀上的差異。K-means 顯示光譜群集分布，而 Random Forest 則根據訓練樣本輸出明確的土地覆蓋類別。

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

task1_dir = r"c:\Users\Ben\Documents\NTU\RSGI\homework11\outputs\Task1"
task2_dir = r"c:\Users\Ben\Documents\NTU\RSGI\homework11\outputs\Task2"

kmeans_map = np.load(os.path.join(task1_dir, "kmeans_label_map.npy"))
rf_map = np.load(os.path.join(task2_dir, "rf_class_map.npy"))

kmeans_plot = np.ma.masked_invalid(kmeans_map)
rf_plot = np.ma.masked_invalid(rf_map)

rf_colors = ["#0077BE", "#228B22", "#DAA520", "#CD853F", "#808080"]
rf_cmap = ListedColormap(rf_colors)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

im0 = axes[0].imshow(kmeans_plot, cmap="tab10")
axes[0].set_title("K-means Classification (K=5)")
axes[0].axis("off")
fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04, label="Cluster ID")

im1 = axes[1].imshow(rf_plot, cmap=rf_cmap, vmin=0, vmax=4)
axes[1].set_title("Random Forest Classification")
axes[1].axis("off")
cbar = fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04, ticks=range(5), label="Class")
cbar.ax.set_yticklabels(class_names)

plt.tight_layout()
plt.show()

fig.savefig(os.path.join(task2_dir, "kmeans_vs_rf_comparison.png"), dpi=300, bbox_inches="tight")
print("Saved: kmeans_vs_rf_comparison.png")

### Brief discussion:
---
    在 Random Forest 的特徵重要性中，B11（SWIR1，短波紅外）排名最高。這是合理的，因為 SWIR1 對地表水分含量、裸露土壤與岩石表面非常敏感。秀林/太魯閣研究區以山區森林、河道、水體、裸地與崩塌地為主，這些地物在 B11 上具有明顯差異。水體在 SWIR 波段反射率很低，森林會受到植被含水量影響，而裸地與崩塌地通常具有較高的 SWIR 反射率。因此，B11 能有效協助模型區分水體、森林、裸地/崩塌地與建成區，是本次分類中最重要的波段。

    另外，從kmeans_cluster_mean_spectrum圖片中 K-means 各群集的平均光譜曲線也可以看出，B11 對區分不同地物類型具有重要作用。相較於 B02、B03、B04 等可見光波段，不同 cluster 在 B11 的反射率差異更明顯。例如部分群集在 B11 上維持較高反射，可能對應裸地、崩塌地或建成區；而水體或陰影類型在 B11 上通常反射率較低。因此，B11 不只在 Random Forest 的 feature importance 中排名最高，也能從 K-means 光譜曲線中觀察到它對五個群集的分離能力較強。

---
---
## Task 3 - Accuracy Assessment and Independent Validation
---
---
本任務評估 Random Forest 土地覆蓋分類結果的可靠度，分為內部驗證與外部驗證兩部分。內部驗證使用訓練資料切分出的 test set，檢查五類土地覆蓋分類的整體準確率與各類別表現；外部驗證則使用 SWCB 官方新生崩塌地資料，比對 RF 的 Bare/Landslide 類別與官方崩塌地之間的空間重疊程度。

工作流程如下：

1. 使用訓練好的 Random Forest 模型預測 test set，產生 `y_pred_test`
2. 建立 confusion matrix heatmap，觀察五類土地覆蓋之間的混淆情形
3. 輸出 classification report，檢查 precision、recall、F1-score 與 support
4. 比較 OOB accuracy 與 Test accuracy，評估模型驗證結果是否穩定
5. 比較 Macro avg F1 與 Weighted avg F1，判斷少數類別是否被整體指標稀釋
6. 讀取 SWCB 官方新生崩塌地 KML，並裁切到 Taroko study area
7. 將 SWCB polygon rasterize 成與 RF classification map 相同大小的二元 mask
8. 將 RF 的 Bare/Landslide 類別視為可能崩塌區，與 SWCB mask 進行比較
9. 計算 Recall、Precision 與 IoU 等空間重疊指標
10. 繪製 TP/FN/FP overlay 圖，分析重疊區、漏判區與誤判區的空間分布

### Part A - Confusion Matrix + Classification Report + OOB/Test Accuracy
---
使用 Random Forest 模型預測 test set，並建立 confusion matrix 觀察五類土地覆蓋之間的分類混淆情形。接著輸出 classification report 與 OOB/Test accuracy，並比較 Macro avg F1 和 Weighted avg F1 的差距，以評估模型整體表現與少數類別是否被加權平均稀釋。

In [ ]:
# Task 3 Part A: Confusion Matrix + Classification Report + OOB/Test Accuracy
import os
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

output_dir = r"c:\Users\Ben\Documents\NTU\RSGI\homework11\outputs\Task3"
os.makedirs(output_dir, exist_ok=True)

# 1) Predict test set
y_pred_test = rf.predict(X_te)

# 2) Confusion matrix
cm = confusion_matrix(y_te, y_pred_test)

fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
disp.plot(cmap="Blues", values_format="d", ax=ax, colorbar=True)
ax.set_title("Classification Confusion Matrix")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

fig.savefig(os.path.join(output_dir, "confusion_matrix.png"), dpi=300, bbox_inches="tight")

# 3) Classification report
report_text = classification_report(
    y_te,
    y_pred_test,
    target_names=class_names
)

report_dict = classification_report(
    y_te,
    y_pred_test,
    target_names=class_names,
    output_dict=True
)

df_report = pd.DataFrame(report_dict).T
df_report.to_csv(os.path.join(output_dir, "classification_report.csv"))

with open(os.path.join(output_dir, "classification_report.txt"), "w", encoding="utf-8") as f:
    f.write(report_text)

# 4) OOB/Test accuracy + Macro/Weighted F1 gap
oob_acc = rf.oob_score_
test_acc = rf.score(X_te, y_te)
macro_f1 = report_dict["macro avg"]["f1-score"]
weighted_f1 = report_dict["weighted avg"]["f1-score"]
f1_gap = weighted_f1 - macro_f1

summary_df = pd.DataFrame({
    "metric": [
        "OOB accuracy",
        "Test accuracy",
        "Macro avg F1",
        "Weighted avg F1",
        "Weighted-Macro F1 gap"
    ],
    "value": [
        oob_acc,
        test_acc,
        macro_f1,
        weighted_f1,
        f1_gap
    ]
})

display(summary_df)
summary_df.to_csv(os.path.join(output_dir, "partA_accuracy_summary.csv"), index=False)

### Brief discussion
---
    OOB accuracy 為 0.8344，Test accuracy 為 0.8298，兩者差距僅約 0.0046，表示 Random Forest 的 OOB 內部驗證與 20% hold-out test set 得到非常接近的結果。這代表模型整體表現相對穩定，沒有明顯過度樂觀的準確率估計。

    Macro avg F1 為 0.73，Weighted avg F1 為 0.82，兩者差距為 0.09，大於 0.03。這表示分類表現受到類別不平衡與少數類別表現較差的影響。Weighted avg 會依照 support 加權，因此 Water、Cropland 等樣本較多或較容易分類的類別會拉高整體分數；相較之下，Built-up 的 support 只有 65，F1-score 僅 0.30，Bare 的 F1-score 也只有 0.68，這些較難分類的類別在 weighted average 中被稀釋。因此，雖然整體 Test accuracy 約 83%，但少數類別的分類可靠度仍需謹慎解讀。

### Part B
---
本段進行 Task 3 Part B 的 SWCB 官方崩塌地外部驗證。首先讀取 `20240802新生崩塌地.kml`，解析其中的崩塌地 polygon，並裁切到秀林/太魯閣研究區範圍內。接著將 SWCB polygon rasterize 成與 Random Forest classification map 相同大小的二元 mask，其中 `1` 代表官方崩塌地，`0` 代表非崩塌地。

為了評估模型對崩塌/裸地的偵測能力，本分析將 RF 分類結果中的 `Bare` 類別視為可能的 landslide/bare-ground prediction，並與 SWCB mask 進行像素層級比較。比較結果包含三個指標：Recall 表示官方崩塌地中有多少比例被 RF 偵測到，Precision 表示 RF 判定為 Bare/Landslide 的像素中有多少被 SWCB 官方資料確認，IoU 則表示兩者空間重疊程度。

最後建立 TP/FN/FP overlay 圖，視覺化 RF 與 SWCB 的空間差異。其中綠色代表 TP，也就是 RF 與 SWCB 都判定為崩塌/裸地的區域；紅色代表 FN，表示 SWCB 標示為崩塌但 RF 沒有偵測到；黃色代表 FP，表示 RF 判定為 Bare/Landslide，但 SWCB 官方資料中沒有標示為新生崩塌地。此圖可用來分析模型漏判與誤判的位置，並輔助討論 Sentinel-2 解析度、地物定義差異、混合像素與影像日期差異所造成的不確定性。

In [ ]:
import os
import geopandas as gpd
import xml.etree.ElementTree as ET
from shapely.geometry import Polygon, box

input_dir = r"c:\Users\Ben\Documents\NTU\RSGI\homework11\data"
swcb_path = os.path.join(input_dir, "20240802新生崩塌地.kml")

tree = ET.parse(swcb_path)
ns = {"kml": "http://www.opengis.net/kml/2.2"}

polygons = []
for pm in tree.getroot().findall(".//kml:Placemark", ns):
    coords_el = pm.find(".//kml:coordinates", ns)
    if coords_el is not None:
        coords = []
        for pt in coords_el.text.strip().split():
            lon, lat, *_ = pt.split(",")
            coords.append((float(lon), float(lat)))
        if len(coords) >= 3:
            polygons.append(Polygon(coords))

gdf_swcb = gpd.GeoDataFrame(geometry=polygons, crs="EPSG:4326")

TAROKO_BBOX = [121.40, 24.10, 121.80, 24.25]
study_box = box(*TAROKO_BBOX)

gdf_clipped = gdf_swcb[gdf_swcb.intersects(study_box)].copy()

print(f"Total SWCB polygons: {len(gdf_swcb)}")
print(f"SWCB landslide polygons in study area: {len(gdf_clipped)}")

In [ ]:
from rasterio.features import rasterize
from rasterio.transform import from_bounds

# Build transform matching your classification map
transform = from_bounds(*TAROKO_BBOX, w, h)

# Rasterize: 1 = SWCB landslide, 0 = not landslide
swcb_mask = rasterize(
    [(geom, 1) for geom in gdf_clipped.geometry],
    out_shape=(h, w),
    transform=transform,
    fill=0,
    dtype='uint8',
)
print(f"SWCB landslide pixels: {swcb_mask.sum()}")

In [ ]:
# Your classification: "Bare/Landslide" class (class_id = 3)
rf_landslide = (class_map == 3).astype(int)  # adjust class_id as needed

# Pixel-level comparison
intersection = np.sum((rf_landslide == 1) & (swcb_mask == 1))
union = np.sum((rf_landslide == 1) | (swcb_mask == 1))

# Recall: how much of SWCB landslide did you detect?
recall = intersection / swcb_mask.sum() if swcb_mask.sum() > 0 else 0
# Precision: how much of your "landslide" is confirmed by SWCB?
precision = intersection / rf_landslide.sum() if rf_landslide.sum() > 0 else 0
# IoU (Intersection over Union)
iou = intersection / union if union > 0 else 0

print(f"Recall (detection rate):  {recall:.3f}")
print(f"Precision (commission):   {precision:.3f}")
print(f"IoU (Jaccard index):      {iou:.3f}")

In [ ]:
# Create overlay map: TP (green), FN (red), FP (yellow)
overlay = np.zeros((h, w, 3), dtype=np.uint8)
overlay[(rf_landslide == 1) & (swcb_mask == 1)] = [0, 200, 0]    # TP: green
overlay[(rf_landslide == 0) & (swcb_mask == 1)] = [200, 0, 0]    # FN: red (missed)
overlay[(rf_landslide == 1) & (swcb_mask == 0)] = [200, 200, 0]  # FP: yellow

plt.figure(figsize=(12, 10))
plt.imshow(overlay)
plt.title('RF vs SWCB Landslide Overlay (Green=TP, Red=FN, Yellow=FP)')
plt.show()
plt.imsave(os.path.join(output_dir, "rf_swcb_overlay.png"), overlay)
print("Saved: rf_swcb_overlay.png")

In [ ]:
# Task 3 Part B: Spatial overlap metrics table
import os
import pandas as pd

output_dir = r"c:\Users\Ben\Documents\NTU\RSGI\homework11\outputs\Task3"
os.makedirs(output_dir, exist_ok=True)

overlap_metrics = pd.DataFrame({
    "Metric": ["Recall", "Precision", "IoU"],
    "Value": [recall, precision, iou],
    "Meaning": [
        "Fraction of SWCB landslide pixels detected by RF Bare/Landslide",
        "Fraction of RF Bare/Landslide pixels confirmed by SWCB",
        "Intersection over Union between RF Bare/Landslide and SWCB landslide"
    ]
})

display(overlap_metrics)

overlap_metrics.to_csv(
    os.path.join(output_dir, "swcb_overlap_metrics.csv"),
    index=False
)

print("Saved: swcb_overlap_metrics.csv")

### Brief discussion
---
    外部驗證結果顯示，RF Bare/Landslide 類別與 SWCB 官方新生崩塌地資料的空間重疊程度偏低。Recall 為 0.213，表示官方崩塌地中約 21.3% 被 RF 的 Bare 類別偵測到；但 Precision 僅為 0.002，IoU 也僅為 0.002，代表 RF 判定為 Bare 的大量像素並非 SWCB 標註的新生崩塌地。

    造成差異的主要原因是兩者的類別定義不同。本研究中的 Bare 類別包含裸露土壤、河床、道路、海岸、岩壁與可能的崩塌地，但 SWCB 資料只包含官方判釋的新生崩塌地。因此，RF Bare 類別會產生大量 false positives。其次，Sentinel-2 影像解析度為 20 m，而 SWCB 崩塌地資料可能依據較高解析度影像判釋，小型或狹長崩塌地在 Sentinel-2 像素中容易與森林、陰影或裸地混合。此外，本次 Sentinel-2 影像日期為 2024-08-27，而 SWCB 資料日期為 2024-08-02，兩者存在時間差，也可能造成部分崩塌範圍與地表狀態不一致。

    從 RF 與 SWCB 的 overlay 圖可以觀察到，重疊區域主要集中在影像右側偏中下方，也就是靠近東側海岸與立霧溪下游附近的山坡/河谷邊緣區域。這些區域可能具有較明顯的裸露土壤或崩塌光譜特徵，因此較容易被 RF 的 Bare 類別偵測到。相較之下，研究區內部山區雖然也有零星 SWCB 崩塌地，但許多位置可能受到植被、陰影、雲遮蔽或 20 m 混合像素影響，導致 RF 未能完整偵測，形成較多 FN。



---
---
## Task 4 - AI Classification Report
---
---
本任務根據 Random Forest 土地覆蓋分類結果計算各類別面積統計，並使用 LLM 生成災後土地覆蓋分析報告。報告內容整合分類面積、模型準確率與 SWCB 崩塌地外部驗證結果，用於說明秀林/太魯閣地區震後土地覆蓋概況、裸地/崩塌分布與分析不確定性。

工作流程如下：

1. 讀取 Random Forest classification map `class_map`
2. 以 20 m 解析度計算每個像素面積，換算各類別面積（hectare）
3. 統計 Water、Forest、Cropland、Bare/Landslide 與 Built-up 的 pixel count、area_ha 與 percentage
4. 將面積統計結果輸出為表格與 `class_area_stats.csv`
5. 將面積統計、Test accuracy、SWCB IoU 與研究背景整理成 prompt
6. 使用 LLM 生成中文 300-500 字的災後土地覆蓋分析報告
7. 檢查 LLM 報告是否有 AI hallucination，例如面積、百分比、accuracy 或 IoU 是否錯誤
8. 評估 LLM 對不確定性的描述是否合理，例如是否提到解析度、類別定義差異、混合像素與時間差
9. 補充或修正 LLM 報告不足之處，並撰寫 critical evaluation


### Step 1 - Areas for clasification calculation
---
根據 Random Forest 的 `class_map` 計算各土地覆蓋類別的面積統計。由於分類影像解析度為 20 m，每個像素面積為 400 平方公尺，因此可將每類像素數轉換為公頃，並計算其在有效分類區域中的百分比，作為後續 AI 報告與災後土地覆蓋分析的基礎資料。

In [ ]:
pixel_area_m2 = 20 * 20  # 20m resolution

class_stats = {}
for i, name in enumerate(class_names):
    n_pixels = np.sum(class_map == i)
    area_ha = n_pixels * pixel_area_m2 / 10000
    class_stats[name] = {
        'pixels': int(n_pixels),
        'area_ha': round(area_ha, 1),
        'percentage': round(n_pixels / np.sum(~np.isnan(class_map)) * 100, 1),
    }

import pandas as pd
stats_df = pd.DataFrame(class_stats).T
print(stats_df)

### LLM Report
---
- 本研究以 2024 年 4 月 3 日花蓮地震後的 Sentinel-2 影像，針對秀林鄉/太魯閣周邊進行 Random Forest 土地覆蓋分類。分類結果顯示，研究區主要由森林與水體構成，其中森林面積約 22,797.6 公頃，占 36.1%；水體面積約 22,557.7 公頃，占 35.8%。這反映本研究範圍同時涵蓋中央山脈東側山區與太平洋近海區域。農地面積約 7,027.6 公頃，占 11.1%；建成區面積約 1,873.0 公頃，占 3.0%，顯示人為開發區域相對有限，主要集中於沿海、河谷與道路周邊。

- 裸地/崩塌地分類面積約 8,828.6 公頃，占 14.0%。從空間分布來看，裸地主要分布於山坡、河谷、道路沿線與部分海岸附近，可能包含地震後崩塌、河床裸露、道路邊坡、岩壁與其他裸露地表。Random Forest 的整體測試準確率為 83.0%，顯示模型具有一定分類能力，但不同類別表現並不完全一致，其中建成區與裸地較容易產生混淆。

- 與 SWCB 官方新生崩塌地資料相比，本研究 Bare/Landslide 類別的 IoU 僅約 0.002，表示兩者空間重疊有限。主要原因可能是類別定義不同：本研究的 Bare 類別包含所有裸露地表，而 SWCB 僅標示官方判釋的新生崩塌地。此外，Sentinel-2 20 m 解析度、雲與陰影遮蔽、混合像素，以及影像日期與 SWCB 判釋日期之間的時間差，都會造成不確定性。

- 此分類圖可作為災後初步判釋工具，協助辨識山區裸露地表與可能崩塌熱區。後續若結合道路、聚落、避難所、坡度與水系資料，可用於評估道路中斷風險、避難所可及性，以及災後巡查與資源配置的優先順序。

### LLM output evaluation：
---
1. Are the area numbers in the LLM report correct? (Check for AI hallucination)
    正確，和實際輸出結果比對後可確認。
2. Is the uncertainty description reasonable?
    是合理的。報告有正確指出 RF 的 Bare/Landslide 類別與 SWCB 官方新生崩塌地資料在定義上不同，因此兩者重疊率低並不代表分類完全錯誤。RF 的 Bare 類別包含河床、道路邊坡、海岸裸地、岩石裸露地與可能的崩塌地，但 SWCB 資料只包含官方判釋的新生崩塌地。
3. What would you add or change?
    如果我要寫Report，我會加上feature importance，既然主軸是以rf作為代表，可以在多提供一些相關的資增加報告的完備性。
